# Amazon Nova Forge SDK - Quick Start Guide for SMTJ Serverless

This notebook provides a basic walkthrough of the Amazon Nova Forge SDK for fine-tuning Nova models using SMTJ Serverless.

## What You'll Learn

1. Loading and preparing datasets
2. Fine-tuning a Nova model with SFT (Supervised Fine-Tuning) using SMTJ Serverless
3. Monitoring training progress
4. Running evaluation on your custom data
5. Deploying your model

## Table of Contents
- [Step 1: Import Required Modules](#step-1-import-required-modules)
- [Step 2: Configure Your AWS Resources](#step-2-configure-your-aws-resources)
- [Step 3: Prepare your Dataset](#step-3-prepare-your-dataset)
- [Step 4: Configure Runtime Infrastructure](#step-4-configure-runtime-infrastructure)
- [Step 5: Initialize ForgeTrainer](#step-5-initialize-forgetrainer)
- [Step 6: Start Training](#step-6-start-training)
- [Step 7: Monitor Training Progress](#step-7-monitor-training-progress)
- [Step 8: Run Evaluation](#step-8-evaluate-your-model-after-training-completes)
- [Step 9: Monitor Evaluation Progress](#step-9-monitor-evaluation-progress)
- [Step 10: Deploy Your Model (After Training Completes)](#step-10-deploy-your-model-after-training-completes)

## Prerequisites

- AWS credentials configured
- S3 bucket for data and model artifacts
- IAM permissions for SageMaker and Bedrock
- Nova Forge SDK installed per [its README](https://github.com/aws/nova-forge-sdk/blob/main/README.md#installation)

## Helpful Links
- If you haven't already, please take a look at the `docs/spec.md` file for more information on what parameters you can change in the code below.
- Also visit the `README.md` for a high-level overview of the Nova SDK and its capabilities.

## Step 1: Import Required Modules

In [ ]:
!pip install amzn-nova-forge

In [ ]:
import os

import boto3
from botocore.exceptions import ClientError, NoCredentialsError, ProfileNotFound


def load_credentials(profile=None):
    """
    Load AWS credentials with fallback behavior.

    Args:
        profile (str, optional): AWS profile name. If provided, loads from credentials file.
                               If None, uses current authenticated AWS session.

    Returns:
        dict: Dictionary containing AWS credentials and region

    Raises:
        RuntimeError: If credential loading fails
    """
    if profile:
        # Try loading from credentials file
        try:
            session = boto3.Session(profile_name=profile)
            credentials = session.get_credentials()

            if not credentials:
                raise RuntimeError(f"No credentials found for profile '{profile}'")

        except ProfileNotFound:
            raise RuntimeError(f"Profile '{profile}' not found in credentials file")
        except Exception as e:
            raise RuntimeError(f"Failed to load credentials from file: {e}")

    else:
        # Try loading from current authenticated session
        try:
            session = boto3.Session()
            credentials = session.get_credentials()

            if not credentials:
                raise RuntimeError("No credentials found in current AWS session")

        except NoCredentialsError:
            raise RuntimeError("No AWS credentials configured")
        except Exception as e:
            raise RuntimeError(f"Failed to load credentials from current session: {e}")

        # Validate credentials by making a test call
    try:
        sts_client = session.client("sts")
        sts_client.get_caller_identity()
    except ClientError as e:
        raise RuntimeError(f"Invalid AWS credentials: {e}")
    except Exception as e:
        raise RuntimeError(f"Failed to validate credentials: {e}")

    return {
        "aws_access_key_id": credentials.access_key,
        "aws_secret_access_key": credentials.secret_key,
        "aws_session_token": credentials.token,
        "region_name": session.region_name or "us-east-1",
    }

In [ ]:
load_credentials()
print("✅ AWS credentials validated successfully!")

In [ ]:
# Core imports — the modular service classes are the recommended API
from amzn_nova_forge import *

print("✅ SDK imported successfully!")

## Step 2: Configure Your AWS Resources

In [ ]:
# TODO: Update these values for your environment
S3_BUCKET = "<your-s3-bucket>"  # TODO: Replace with your S3 bucket
S3_DATA_PATH = f"s3://{S3_BUCKET}/demo/input"
S3_OUTPUT_PATH = f"s3://{S3_BUCKET}/demo/output"

print(f"Data Path: {S3_DATA_PATH}")
print(f"Output Path: {S3_OUTPUT_PATH}")

## Step 3: Prepare Your Dataset

The SDK supports three formats: **JSONL**, **JSON**, and **CSV**. This example uses JSONL.

In [ ]:
# Create sample training data
import json

sample_data = [
    {
        "question": "What is machine learning?",
        "answer": "Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.",
    },
    {
        "question": "Explain what AWS is.",
        "answer": "AWS (Amazon Web Services) is a comprehensive cloud computing platform that provides on-demand computing resources and services.",
    },
    {
        "question": "What is Python used for?",
        "answer": "Python is a versatile programming language used for web development, data analysis, artificial intelligence, scientific computing, and automation.",
    },
] * 100

# Save sample data locally
with open("training_data.jsonl", "w") as f:
    for item in sample_data:
        f.write(json.dumps(item) + "\n")

print("✅ Sample data created: training_data.jsonl")

### Load, Transform, and Validate the Dataset

In [ ]:
# Initialize dataset loader
loader = JSONLDatasetLoader()

# Load the data
loader.load("training_data.jsonl")

# Preview the data
print("\n📊 Dataset Preview:")
loader.show(n=3)

In [ ]:
# Transform data for Nova model training
loader.transform(
    method=TransformMethod.SCHEMA,
    training_method=TrainingMethod.SFT_LORA,
    model=Model.NOVA_LITE_2,
    column_mappings={"question": "question", "answer": "answer"},
)

print("✅ Data transformed to Converse format")
print("\n📊 Transformed Data Preview:")

loader.show(n=5)

In [ ]:
# Validates transformed data for the method and model combination. Prints out a "Validation completed" message if successful.
loader.validate(
    method=ValidateMethod.SCHEMA,
    training_method=TrainingMethod.SFT_LORA,
    model=Model.NOVA_LITE_2,
)

### Split and Save Dataset

In [ ]:
# Split into train/validation sets
train_loader, val_loader, _ = loader.split(train_ratio=0.7, val_ratio=0.2, test_ratio=0.1)

# Save datasets
# For production, upload to S3:
train_path = train_loader.save(f"{S3_DATA_PATH}/train.jsonl")
val_path = val_loader.save(f"{S3_DATA_PATH}/val.jsonl")

print(f"\n✅ Training data saved to: {train_path}")
print(f"✅ Validation data saved to: {val_path}")

## Step 4: Configure Runtime Infrastructure

Choose between:
- **SMTJRuntimeManager**: For SageMaker Training Jobs
- **SMTJServerlessRuntimeManager**: For SageMaker Serverless Training Jobs (no infrastructure management)
- **SMHPRuntimeManager**: For SageMaker HyperPod clusters
- **BedrockRuntimeManager**: For Amazon Bedrock (fully managed)

In [ ]:
from amzn_nova_forge.model import Platform

In [ ]:
# Change the package_group_name to desired name, this creates a SageMaker Hub Resource to store resources and training artifacts
runtime = SMTJServerlessRuntimeManager(model_package_group_name="test-package")

platform = Platform.SMTJServerless

print("✅ Runtime configured for SMTJ Serverless")

## Step 5: Initialize ForgeTrainer

### Create MLFlow monitor

In [ ]:
# Create MLflow monitor to monitor metrics, this is optional
mlflow_monitor = MLflowMonitor(
    tracking_uri="arn:aws:sagemaker:<region>:<account-id>:mlflow-app/<app-id>",
    experiment_name="nova-customization-experiment",  # replace with experiment name
    run_name="nova-lite2-sft-run-1",  # replace with run name
)

# Generate presigned URL to access MLflow UI directly
mlflow_url = mlflow_monitor.get_presigned_url()
print(f"Access MLflow UI at: {mlflow_url}")

# mlflow_monitor = MLflowMonitor() # uses default mlflow app, if it exists

# mlflow_monitor = MLflowMonitor(
#     experiment_name="nova-customization-experiment", # replace with experiment name
#     run_name="nova-lite2-sft-run-1" # replace with run name
# ) # uses default mlflow app, if it exists

In [ ]:
# Create trainer
trainer = ForgeTrainer(
    model=Model.NOVA_LITE_2,  # Choose your Nova model
    # model_s3_path="s3_path_of_checkpoint"  # for specifying iterative training
    method=TrainingMethod.SFT_LORA,  # Training method
    infra=runtime,  # Runtime configuration
    training_data_s3_path=train_path,  # Training data path
    config=ForgeConfig(
        output_s3_path=S3_OUTPUT_PATH,  # Output path for artifacts
        mlflow_monitor=mlflow_monitor,  # optional
    ),
)
print("✅ ForgeTrainer initialized")
print(f"   Model: Nova Lite 2.0")
print(f"   Method: SFT with LoRA")

## Step 6: Start Training

In [ ]:
# Define training hyperparameters - You can edit these by navigating to the Nova Customization public documentation, linked in the ../docs/spec.md document.
training_config = {
    "lr": 5e-6,  # Learning rate
    "warmup_steps": 100,  # Warmup steps
    "global_batch_size": 64,  # Batch size
    "max_length": 8192,  # Max sequence length,
}

# Start training
training_result = trainer.train(job_name="<your-job-name>", overrides=training_config)

print("\n🚀 Training job started!")
print(training_result)
print(
    f"   📍 Checkpoint URI where the model will be saved: {training_result.model_artifacts.checkpoint_s3_path}"
)
print(f"   🆔 Job ID: {training_result.job_id}")
print(f"   📂 Output Path: {training_result.model_artifacts.output_s3_path}")

# Save job ID for later
job_id = training_result.job_id
escrow_uri = training_result.model_artifacts.checkpoint_s3_path
output_path = training_result.model_artifacts.output_s3_path

In [ ]:
training_result_path = training_result.dump()
print("✅ Training result saved")

In [ ]:
loaded_training_result = TrainingResult.load(training_result_path)
print(f"✅ Loaded — Job ID: {loaded_training_result.job_id}")

## Step 7: Monitor Training Progress

In [ ]:
job_id = loaded_training_result.job_id

In [ ]:
monitor = CloudWatchLogMonitor.from_job_id(job_id=job_id, platform=Platform.SMTJServerless)
monitor.show_logs(limit=100)

In [ ]:
monitor.plot_metrics(training_method=TrainingMethod.SFT_LORA)

## Step 8: Evaluate Your Model (After Training Completes)

Evaluation jobs allow you to test your customized model against pre-set or custom benchmarks.

In [ ]:
evaluator = ForgeEvaluator(
    model=Model.NOVA_LITE_2,
    infra=runtime,
    config=ForgeConfig(
        output_s3_path=S3_OUTPUT_PATH,
    ),
)

In [ ]:
byod_eval_result = evaluator.evaluate(
    job_name="serverless-eval-byod",
    eval_task=EvaluationTask.GEN_QA,
    task_config=EvalTaskConfig(
        override_data_s3_path="s3://<data-s3-bucket>/nova-customization/gen_qa.jsonl",  # TODO: Replace with your data path
    ),
    # model_path='s3://customer-escrow-<your-model-ckpt-bucket>/your-model-path/'  # TODO: Replace with your trained model path
    overrides={"max_new_tokens": 2048},
)

print("  📍 Bring Your Own Data Job ID: ", byod_eval_result.job_id)
print("  📂 Bring Your Own Data Output Path:", byod_eval_result.eval_output_path)

In [ ]:
evaluation_result_path = byod_eval_result.dump()
print("✅ Evaluation result saved")

In [ ]:
loaded_evaluation_result = EvaluationResult.load(evaluation_result_path)
print(f"✅ Loaded — Job ID: {loaded_evaluation_result.job_id}")

## Step 9: Monitor Evaluation Progress

In [ ]:
job_id = loaded_evaluation_result.job_id

In [ ]:
monitor = CloudWatchLogMonitor.from_job_id(job_id=job_id, platform=Platform.SMTJServerless)
monitor.show_logs(limit=100)

## Step 10: Deploy Your Model (After Training Completes)

Once training is complete, deploy your model to Amazon Bedrock or SageMaker

In [ ]:
deployer = ForgeDeployer(
    region=os.environ.get("AWS_DEFAULT_REGION", "us-east-1"),
    model=Model.NOVA_LITE_2,
)

# Deploy to SageMaker
# deployment_result = deployer.deploy(
#     model_artifact_path='s3://<your-checkpoint-s3-path>',
#     deploy_platform=DeployPlatform.SAGEMAKER,
#     unit_count=1,
#     endpoint_name="<your-endpoint-name>",
#     sagemaker_environment_variables={
#         "CONTEXT_LENGTH": "8000",
#         "MAX_CONCURRENCY": "32",
#     },
# )

# Bedrock deployment is also supported
deployment_result = deployer.deploy(
    model_artifact_path="s3://<your-checkpoint-s3-path>",
    deploy_platform=DeployPlatform.BEDROCK_OD,
    endpoint_name="<your-endpoint-name>",
)

print("\n🚀 Model deployment started!")
print(f"   Endpoint Name: {deployment_result.endpoint.endpoint_name}")
print(f"   Status: {deployment_result.status}")